<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/YOLO26n_MOT_Congestion_Grafica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medición de congestión vehicular con YOLO26n + MOT (visualización gráfica)

Este notebook procesa **un video de tráfico**. YOLO26n detecta vehículos y ByteTrack (MOT) conserva un ID temporal por vehículo. Se calculan por fotograma:

- Vehículos visibles dentro de cada **zona** (una por calzada) y en total.
- Estado de cada vehículo: **FLUIDO, LENTO o DETENIDO**, con una velocidad normalizada por perspectiva (alturas de caja por segundo).
- Índice de congestión 0-100 = ocupación + lentitud + proporción detenida, suavizado y con histéresis; nivel **Baja, Moderada o Alta**, global y por zona.

**El video de salida se vuelve más gráfico conforme aumenta la congestión:**

| Elemento | Qué muestra |
|---|---|
| Zonas coloreadas | Cada calzada se tiñe de verde / ámbar / rojo según su propio nivel; en ALTA el relleno "late". |
| Cajas por estado | Esquinas verdes = fluido, caja ámbar = lento, caja roja rellena + `STOP` = detenido. Estela de trayectoria. |
| Mapa de calor | Resplandor amarillo→rojo donde se acumulan vehículos lentos o detenidos (las colas). |
| Panel lateral | Velocímetro del índice, conteos por estado, barra de composición, barras por zona y mini-gráfica de los últimos 15 s. |
| Alertas | Borde ámbar + banner "TRÁFICO DENSO" en moderada; borde rojo pulsante + banner de **ALERTA** con duración cuando la congestión ALTA se sostiene (global o en una zona). |

> **Importante:** sin calibración de cámara (homografía/escala) no se obtienen km/h. El índice es una estimación visual para comparar segmentos del mismo video; zonas, capacidades y umbrales deben ajustarse a la cámara y al encuadre.


## 1. Instalar dependencias

In [ ]:
!pip -q install -U ultralytics tqdm

## 2. Cargar video

In [ ]:
from google.colab import files
uploaded = files.upload()  # Selecciona un video MP4, AVI o MOV
VIDEO_PATH = next(iter(uploaded))
print("Video:", VIDEO_PATH)

## 3. Configuración

In [ ]:
from pathlib import Path
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, math, os
from collections import deque
from ultralytics import YOLO
from tqdm.auto import tqdm
from IPython.display import Video, Image, display

MODEL_NAME = "yolo26n.pt"
TRACKER = "bytetrack.yaml"       # MOT: ByteTrack
CONFIDENCE = 0.25
# 960 detecta mucho mejor los vehículos lejanos y las motos de este encuadre
# (≈25 vehículos/fotograma frente a ≈14 con 640). En CPU es ~1.7x más lento.
IMAGE_SIZE = 960

# Clases COCO de vehículos: car=2, motorcycle=3, bus=5, truck=7
VEHICLE_CLASSES = [2, 3, 5, 7]

# ---------------------------------------------------------------------------
# ZONAS DE ANÁLISIS (coordenadas para el video de 1280x720 de la autopista).
# Cada calzada se evalúa por separado y se colorea según su propio nivel.
# Las coordenadas se reescalan solas si el video tiene otra resolución.
# Usa ZONES = None para analizar el fotograma completo como una sola zona.
# ---------------------------------------------------------------------------
ZONES_REF_SIZE = (1280, 720)
ZONES = {
    "Calzada izquierda": [(0, 470), (250, 405), (560, 322), (785, 318), (235, 719), (0, 719)],
    "Calzada derecha":   [(785, 318), (1050, 305), (995, 470), (925, 719), (235, 719)],
}
# Vehículos que caben "cómodamente" en cada zona. Por encima de esta cifra la
# ocupación se considera saturada (100 %). Ajusta según el encuadre.
ZONE_CAPACITY = {"Calzada izquierda": 12, "Calzada derecha": 16}
DEFAULT_CAPACITY = 25            # se usa si ZONES = None

# ---------------------------------------------------------------------------
# VELOCIDAD NORMALIZADA POR PERSPECTIVA
# En vez de px/s se usa "alturas de caja por segundo" (bh/s): un auto lejano
# es pequeño y se desplaza pocos píxeles aunque vaya igual de rápido que uno
# cercano. Dividir entre la altura de su caja compensa la perspectiva.
# ---------------------------------------------------------------------------
STOPPED_BH_S = 0.20              # por debajo: DETENIDO
SLOW_BH_S = 1.20                 # por debajo: LENTO; por encima: FLUIDO
FREE_FLOW_BH_S = 4.0             # velocidad de referencia con flujo libre
HISTORY_SECONDS = 1.0            # ventana para medir desplazamiento
MIN_HISTORY_SECONDS = 0.40       # no clasificar hasta tener este historial

# Índice 0-100 = ocupación + lentitud + proporción detenida
W_OCCUPANCY, W_SLOWNESS, W_STOPPED = 0.40, 0.35, 0.25

# Niveles (con histéresis para que el color no parpadee)
LIMIT_MODERATE = 35
LIMIT_HIGH = 60
HYSTERESIS = 4
EMA_ALPHA = 0.08                 # suavizado del índice (0-1; menor = más suave)
ALERT_MIN_SECONDS = 1.5          # ALTA sostenida para disparar la alerta visual

# Elementos gráficos
SHOW_HEATMAP = True              # mapa de calor de vehículos lentos/detenidos
HEAT_DECAY = 0.96                # memoria del mapa de calor por fotograma
HEAT_MAX = 4.0                   # valor que se pinta con el color más intenso
SHOW_TRAILS = True               # estela de trayectoria por vehículo
SPARKLINE_SECONDS = 15           # ventana de la mini-gráfica del panel

OUTPUT_VIDEO = "/content/congestion_yolo26n_mot.mp4"
OUTPUT_CSV = "/content/metricas_congestion.csv"
OUTPUT_PEAK = "/content/pico_congestion.png"

print("GPU disponible:", torch.cuda.is_available())


## 4. Cargar YOLO26n

In [ ]:
model = YOLO(MODEL_NAME)
print("Modelo cargado:", MODEL_NAME)

## 5. Funciones de visualización (panel, velocímetro, mapa de calor, alertas)

In [ ]:
# Paleta BGR (OpenCV)
GREEN = (90, 200, 70)
AMBER = (0, 185, 255)
RED = (50, 40, 235)
GRAY = (190, 190, 190)
WHITE = (255, 255, 255)
DARK = (25, 25, 25)
LEVEL_COLORS = {"BAJA": GREEN, "MODERADA": AMBER, "ALTA": RED}
STATE_COLORS = {"FLUIDO": GREEN, "LENTO": AMBER, "DETENIDO": RED, "NUEVO": GRAY}
FONT = cv2.FONT_HERSHEY_SIMPLEX   # Hershey no admite acentos: los textos del video van sin tildes


def congestion_label(score):
    if score < LIMIT_MODERATE:
        return "BAJA"
    elif score < LIMIT_HIGH:
        return "MODERADA"
    return "ALTA"


def level_with_hysteresis(score, previous):
    """Sube de nivel al cruzar el límite; baja sólo si cae HYSTERESIS puntos por debajo."""
    if previous is None:
        return congestion_label(score)
    if previous == "ALTA":
        return "ALTA" if score >= LIMIT_HIGH - HYSTERESIS else congestion_label(score)
    if previous == "MODERADA":
        if score >= LIMIT_HIGH:
            return "ALTA"
        return "MODERADA" if score >= LIMIT_MODERATE - HYSTERESIS else "BAJA"
    return congestion_label(score)


def congestion_index(n_vehicles, capacity, speeds, n_stopped, n_classified):
    occupancy = min(n_vehicles / max(capacity, 1), 1.0)
    slowness = float(np.clip(1.0 - np.median(speeds) / FREE_FLOW_BH_S, 0, 1)) if speeds else 0.0
    stopped_ratio = n_stopped / n_classified if n_classified else 0.0
    score = 100.0 * (W_OCCUPANCY * occupancy + W_SLOWNESS * slowness + W_STOPPED * stopped_ratio)
    return score, occupancy, slowness, stopped_ratio


def blend_rect(img, p1, p2, color, alpha):
    x1, y1 = max(p1[0], 0), max(p1[1], 0)
    x2, y2 = min(p2[0], img.shape[1]), min(p2[1], img.shape[0])
    if x2 <= x1 or y2 <= y1:
        return
    roi = img[y1:y2, x1:x2]
    block = np.full_like(roi, color)
    cv2.addWeighted(block, alpha, roi, 1 - alpha, 0, dst=roi)


def put_text(img, text, org, scale, color=WHITE, thick=1, shadow=True):
    if shadow:
        cv2.putText(img, text, (org[0] + 1, org[1] + 1), FONT, scale, (0, 0, 0), thick + 2, cv2.LINE_AA)
    cv2.putText(img, text, org, FONT, scale, color, thick, cv2.LINE_AA)


def put_text_centered(img, text, center_x, y, scale, color=WHITE, thick=1):
    (tw, _), _ = cv2.getTextSize(text, FONT, scale, thick)
    put_text(img, text, (int(center_x - tw / 2), int(y)), scale, color, thick)


def draw_corner_box(img, x1, y1, x2, y2, color, t=2):
    L = max(6, int(min(x2 - x1, y2 - y1) * 0.28))
    for (px, py, dx, dy) in [(x1, y1, 1, 1), (x2, y1, -1, 1), (x1, y2, 1, -1), (x2, y2, -1, -1)]:
        cv2.line(img, (px, py), (px + dx * L, py), color, t, cv2.LINE_AA)
        cv2.line(img, (px, py), (px, py + dy * L), color, t, cv2.LINE_AA)


def draw_vehicle(img, box, state, s):
    x1, y1, x2, y2 = box
    color = STATE_COLORS[state]
    if state == "DETENIDO":
        blend_rect(img, (x1, y1), (x2, y2), RED, 0.35)
        cv2.rectangle(img, (x1, y1), (x2, y2), RED, max(2, int(2 * s)), cv2.LINE_AA)
        tag = "STOP"
        (tw, th), _ = cv2.getTextSize(tag, FONT, 0.42 * s, 1)
        cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw + 6, y1), RED, -1)
        cv2.putText(img, tag, (x1 + 3, y1 - 4), FONT, 0.42 * s, WHITE, 1, cv2.LINE_AA)
    elif state == "LENTO":
        blend_rect(img, (x1, y1), (x2, y2), AMBER, 0.15)
        draw_corner_box(img, x1, y1, x2, y2, AMBER, max(2, int(2 * s)))
    else:
        draw_corner_box(img, x1, y1, x2, y2, color, max(1, int(2 * s)))


def draw_trail(img, pts, color, s):
    n = len(pts)
    for k in range(1, n):
        thick = max(1, int(round((1 + 3 * k / n) * s)))
        cv2.line(img, pts[k - 1], pts[k], color, thick, cv2.LINE_AA)


def draw_zones(img, zones_px, zone_levels, zone_scores, pulse, s):
    overlay = img.copy()
    for name, poly in zones_px.items():
        cv2.fillPoly(overlay, [poly], LEVEL_COLORS[zone_levels[name]])
    # La intensidad del relleno crece con la congestión y "late" en ALTA
    any_high = any(lv == "ALTA" for lv in zone_levels.values())
    alpha = 0.16 + (0.14 * pulse if any_high else 0)
    cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, dst=img)
    for name, poly in zones_px.items():
        lv = zone_levels[name]
        cv2.polylines(img, [poly], True, LEVEL_COLORS[lv], max(2, int((3 if lv == "ALTA" else 2) * s)), cv2.LINE_AA)
        # Etiqueta de la zona en su punto inferior-central
        bottom = poly[poly[:, 1] >= poly[:, 1].max() - 2]
        cx = int(bottom[:, 0].mean()) if len(zones_px) > 1 else int(poly[:, 0].mean())
        y = int(poly[:, 1].max() - 14 * s)
        text = f"{name.upper()}: {lv} {zone_scores[name]:.0f}"
        (tw, th), _ = cv2.getTextSize(text, FONT, 0.5 * s, 1)
        cx = int(np.clip(cx, tw // 2 + 12 * s, img.shape[1] - tw // 2 - 12 * s))
        blend_rect(img, (cx - tw // 2 - 8, y - th - 8), (cx + tw // 2 + 8, y + 6), LEVEL_COLORS[lv], 0.85)
        cv2.putText(img, text, (cx - tw // 2, y), FONT, 0.5 * s, WHITE if lv != "MODERADA" else DARK, 1, cv2.LINE_AA)


# Rampa amarillo -> naranja -> rojo para el mapa de calor (sin azules que ensucien la escena)
_ramp = np.linspace(0, 1, 256)[:, None]
HEAT_LUT = np.concatenate([np.zeros_like(_ramp), 210 * (1 - _ramp), 255 - 35 * _ramp], 1).astype(np.uint8)[:, None, :]


def render_heatmap(img, heat, s):
    """Resplandor cálido donde se acumulan vehículos lentos/detenidos."""
    blurred = cv2.GaussianBlur(heat, (0, 0), 5)
    norm = np.clip(blurred / HEAT_MAX, 0, 1)
    if norm.max() < 0.15:
        return
    idx = (norm * 255).astype(np.uint8)
    color = cv2.LUT(cv2.merge([idx, idx, idx]), HEAT_LUT)
    color = cv2.resize(color, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_LINEAR)
    alpha = cv2.resize(np.where(norm > 0.15, (norm - 0.15) / 0.85 * 0.5, 0).astype(np.float32),
                       (img.shape[1], img.shape[0]), interpolation=cv2.INTER_LINEAR)[..., None]
    img[:] = (img * (1 - alpha) + color * alpha).astype(np.uint8)


def draw_gauge(img, center, r, score, level, s):
    cx, cy = center
    t = max(6, int(12 * s))
    segs = [(0, LIMIT_MODERATE, GREEN), (LIMIT_MODERATE, LIMIT_HIGH, AMBER), (LIMIT_HIGH, 100, RED)]
    for a, b, col in segs:
        cv2.ellipse(img, (cx, cy), (r, r), 0, 180 + a * 1.8, 180 + b * 1.8, tuple(int(c * 0.35) for c in col), t, cv2.LINE_AA)
        hi = min(score, b)
        if hi > a:
            cv2.ellipse(img, (cx, cy), (r, r), 0, 180 + a * 1.8, 180 + hi * 1.8, col, t, cv2.LINE_AA)
    # Marcador sobre el arco (en lugar de aguja, para no tapar el número)
    ang = math.radians(180 + np.clip(score, 0, 100) * 1.8)
    mx, my = int(cx + r * math.cos(ang)), int(cy + r * math.sin(ang))
    cv2.circle(img, (mx, my), int(t * 0.95), WHITE, -1, cv2.LINE_AA)
    cv2.circle(img, (mx, my), int(t * 0.55), LEVEL_COLORS[level], -1, cv2.LINE_AA)
    inner = (int(cx + (r - 2.2 * t) * math.cos(ang)), int(cy + (r - 2.2 * t) * math.sin(ang)))
    cv2.line(img, inner, (int(cx + (r - t) * math.cos(ang)), int(cy + (r - t) * math.sin(ang))), WHITE, max(2, int(3 * s)), cv2.LINE_AA)
    put_text_centered(img, f"{score:.0f}", cx, cy - int(r * 0.12), 1.5 * s, WHITE, max(2, int(3 * s)))
    put_text_centered(img, "/ 100", cx, cy + int(4 * s) + int(r * 0.0), 0.4 * s, GRAY, 1)
    # Insignia del nivel
    col = LEVEL_COLORS[level]
    (tw, th), _ = cv2.getTextSize(level, FONT, 0.7 * s, 2)
    bx1, by1 = cx - tw // 2 - int(12 * s), cy + int(16 * s)
    bx2, by2 = cx + tw // 2 + int(12 * s), by1 + th + int(14 * s)
    cv2.rectangle(img, (bx1, by1), (bx2, by2), col, -1)
    cv2.putText(img, level, (cx - tw // 2, by2 - int(7 * s)), FONT, 0.7 * s,
                DARK if level == "MODERADA" else WHITE, 2, cv2.LINE_AA)
    return by2


def draw_hbar(img, x, y, w, h, frac, color):
    cv2.rectangle(img, (x, y), (x + w, y + h), (70, 70, 70), -1)
    cv2.rectangle(img, (x, y), (x + int(w * np.clip(frac, 0, 1)), y + h), color, -1)


def draw_sparkline(img, x, y, w, h, history, now, s):
    blend_rect(img, (x, y), (x + w, y + h), (0, 0, 0), 0.5)
    def ymap(v):
        return int(y + h - (np.clip(v, 0, 100) / 100) * h)
    for a, b, col in [(0, LIMIT_MODERATE, GREEN), (LIMIT_MODERATE, LIMIT_HIGH, AMBER), (LIMIT_HIGH, 100, RED)]:
        blend_rect(img, (x, ymap(b)), (x + w, ymap(a)), col, 0.12)
    pts = [(int(x + w - (now - t) / SPARKLINE_SECONDS * w), ymap(v)) for t, v in history]
    for k in range(1, len(pts)):
        col = LEVEL_COLORS[congestion_label(history[k][1])]
        cv2.line(img, pts[k - 1], pts[k], col, max(2, int(2 * s)), cv2.LINE_AA)
    cv2.rectangle(img, (x, y), (x + w, y + h), (110, 110, 110), 1)


def draw_panel(img, x0, s, d):
    H = img.shape[0]
    pw = img.shape[1] - x0
    blend_rect(img, (x0, 0), (img.shape[1], H), (15, 15, 15), 0.72)
    cv2.line(img, (x0, 0), (x0, H), LEVEL_COLORS[d["level"]], max(2, int(4 * s)))
    pad = int(14 * s)
    cx = x0 + pw // 2
    y = int(28 * s)
    put_text_centered(img, "MONITOR DE CONGESTION", cx, y, 0.5 * s, GRAY, 1)
    put_text_centered(img, f"t = {d['t']:5.1f} s", cx, y + int(20 * s), 0.42 * s, GRAY, 1)

    r = int(pw * 0.36)
    y = draw_gauge(img, (cx, y + int(42 * s) + r), r, d["score"], d["level"], s) + int(26 * s)

    # Conteos por estado
    for label, val, col in [("Vehiculos", d["n"], WHITE), ("Fluidos", d["n_free"], GREEN),
                            ("Lentos", d["n_slow"], AMBER), ("Detenidos", d["n_stop"], RED)]:
        cv2.circle(img, (x0 + pad + 5, y - int(5 * s)), max(3, int(5 * s)), col, -1, cv2.LINE_AA)
        put_text(img, label, (x0 + pad + int(16 * s), y), 0.5 * s, WHITE, 1)
        (tw, _), _ = cv2.getTextSize(str(val), FONT, 0.6 * s, 2)
        put_text(img, str(val), (img.shape[1] - pad - tw, y), 0.6 * s, col, 2)
        y += int(23 * s)

    # Barra apilada de composición del tráfico
    bw = pw - 2 * pad
    total = max(d["n_free"] + d["n_slow"] + d["n_stop"], 1)
    xx = x0 + pad
    for val, col in [(d["n_free"], GREEN), (d["n_slow"], AMBER), (d["n_stop"], RED)]:
        w = int(bw * val / total)
        cv2.rectangle(img, (xx, y), (xx + w, y + int(10 * s)), col, -1)
        xx += w
    y += int(32 * s)

    # Barras por zona
    for name, sc in d["zone_scores"].items():
        lv = d["zone_levels"][name]
        put_text(img, name, (x0 + pad, y), 0.45 * s, WHITE, 1)
        (tw, _), _ = cv2.getTextSize(f"{sc:.0f}", FONT, 0.5 * s, 1)
        put_text(img, f"{sc:.0f}", (img.shape[1] - pad - tw, y), 0.5 * s, LEVEL_COLORS[lv], 1)
        draw_hbar(img, x0 + pad, y + int(6 * s), bw, int(9 * s), sc / 100, LEVEL_COLORS[lv])
        y += int(36 * s)

    put_text(img, f"Ultimos {SPARKLINE_SECONDS} s", (x0 + pad, y), 0.42 * s, GRAY, 1)
    sh = int(min(70 * s, H - y - 22 * s))
    if sh > 20:
        draw_sparkline(img, x0 + pad, y + int(8 * s), bw, sh, d["history"], d["t"], s)


def draw_alert(img, x_max, level, high_for, alert_zones, pulse, s):
    """Borde y banner. Se activa si la escena completa o alguna zona sostiene nivel ALTA."""
    H, W = img.shape[:2]
    global_alert = level == "ALTA" and high_for >= ALERT_MIN_SECONDS
    if global_alert or alert_zones:
        t = int((6 + 8 * pulse) * s)
        cv2.rectangle(img, (0, 0), (W - 1, H - 1), RED, t)
        bh = int(52 * s)
        blend_rect(img, (0, 0), (x_max, bh), RED, 0.55 + 0.3 * pulse)
        # Triángulo de advertencia
        tx, ty, ts = int(22 * s), int(8 * s), int(36 * s)
        tri = np.array([(tx + ts // 2, ty), (tx, ty + ts), (tx + ts, ty + ts)], np.int32)
        cv2.fillPoly(img, [tri], (0, 230, 255), cv2.LINE_AA)
        put_text_centered(img, "!", tx + ts // 2, ty + ts - int(6 * s), 0.75 * s, DARK, 2)
        if global_alert:
            msg = f"ALERTA: CONGESTION ALTA   ({high_for:4.1f} s sostenida)"
        else:
            name, dur = max(alert_zones.items(), key=lambda kv: kv[1])
            extra = f" +{len(alert_zones) - 1}" if len(alert_zones) > 1 else ""
            msg = f"ALERTA: CONGESTION ALTA EN {name.upper()}{extra}  ({dur:4.1f} s)"
        put_text(img, msg, (tx + ts + int(14 * s), int(35 * s)), 0.72 * s, WHITE, 2)
    elif level == "MODERADA":
        cv2.rectangle(img, (0, 0), (W - 1, H - 1), AMBER, max(3, int(4 * s)))
        bh = int(38 * s)
        blend_rect(img, (0, 0), (x_max, bh), AMBER, 0.45)
        put_text(img, "TRAFICO DENSO - congestion moderada", (int(18 * s), int(26 * s)), 0.65 * s, WHITE, 2)


## 6. Procesar el video con MOT y calcular congestión

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or not np.isfinite(fps) or fps <= 0:
    fps = 25.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if width <= 0 or height <= 0:
    cap.release()
    raise RuntimeError("El video no reporta dimensiones válidas.")

S = height / 720                         # escala de textos y grosores
PANEL_W = int(250 * S)
PANEL_X = width - PANEL_W

# Zonas reescaladas a la resolución real del video
if ZONES:
    sx, sy = width / ZONES_REF_SIZE[0], height / ZONES_REF_SIZE[1]
    zones_px = {k: np.array([(int(x * sx), int(y * sy)) for x, y in v], np.int32) for k, v in ZONES.items()}
    capacities = {k: ZONE_CAPACITY.get(k, DEFAULT_CAPACITY) for k in ZONES}
else:
    zones_px = {"Escena": np.array([(0, 0), (PANEL_X - 1, 0), (PANEL_X - 1, height - 1), (0, height - 1)], np.int32)}
    capacities = {"Escena": DEFAULT_CAPACITY}
total_capacity = sum(capacities.values())

def zone_of(point):
    for name, poly in zones_px.items():
        if cv2.pointPolygonTest(poly, (float(point[0]), float(point[1])), False) >= 0:
            return name
    return None

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))
if not writer.isOpened():
    cap.release()
    raise RuntimeError("No se pudo crear el video de salida.")

HEAT_DS = 4                              # el mapa de calor se calcula a 1/4 de resolución
heat = np.zeros((height // HEAT_DS + 1, width // HEAT_DS + 1), np.float32)
track_history, trails = {}, {}
score_ema, zone_ema = None, {k: None for k in zones_px}
level, zone_levels = None, {k: None for k in zones_px}
high_since = None
zone_high_since = {k: None for k in zones_px}
spark = deque()
peak = (-1, None, None)
ema_k = 0
rows = []
frame_idx = 0
names = model.names

pbar = tqdm(total=total_frames if total_frames > 0 else None, desc="Analizando video")

while True:
    ok, frame = cap.read()
    if not ok:
        break
    timestamp = frame_idx / fps

    result = model.track(frame, persist=True, tracker=TRACKER, classes=VEHICLE_CLASSES,
                         conf=CONFIDENCE, imgsz=IMAGE_SIZE, verbose=False)[0]

    vehicles = []   # (tid, box, zone, state, speed_bh_s)
    boxes = result.boxes
    if boxes is not None and len(boxes) > 0 and boxes.id is not None:
        xyxy = boxes.xyxy.cpu().numpy()
        ids = boxes.id.int().cpu().numpy()
        for box, track_id in zip(xyxy, ids):
            x1, y1, x2, y2 = map(int, box)
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            zone = zone_of((cx, cy))
            if zone is None:
                continue
            tid = int(track_id)
            bh = max(y2 - y1, 1)
            hist = track_history.setdefault(tid, deque())
            hist.append((timestamp, cx, cy))
            while hist and timestamp - hist[0][0] > HISTORY_SECONDS:
                hist.popleft()
            trail = trails.setdefault(tid, deque(maxlen=int(fps * 0.8)))
            trail.append((cx, cy))

            state, speed = "NUEVO", None
            t0, x0, y0 = hist[0]
            if timestamp - t0 >= MIN_HISTORY_SECONDS:
                speed = math.hypot(cx - x0, cy - y0) / (timestamp - t0) / bh   # alturas de caja / s
                state = "DETENIDO" if speed < STOPPED_BH_S else "LENTO" if speed < SLOW_BH_S else "FLUIDO"
            vehicles.append((tid, (x1, y1, x2, y2), zone, state, speed))

    # ---- Métricas globales y por zona ----
    def metrics_for(vs, capacity):
        speeds = [v[4] for v in vs if v[4] is not None]
        n_stop = sum(v[3] == "DETENIDO" for v in vs)
        return congestion_index(len(vs), capacity, speeds, n_stop, len(speeds))

    raw_score, occupancy, slowness, stopped_ratio = metrics_for(vehicles, total_capacity)
    # Arranque: mientras no hay historial de velocidad el suavizado se reinicia
    # y después usa un alfa decreciente para no "arrastrar" el valor inicial.
    if timestamp < MIN_HISTORY_SECONDS:
        ema_k = 0
    ema_k += 1
    alpha = max(EMA_ALPHA, 1.0 / ema_k)
    score_ema = raw_score if score_ema is None else (1 - alpha) * score_ema + alpha * raw_score
    level = level_with_hysteresis(score_ema, level)
    zone_raw = {}
    for name in zones_px:
        zs = metrics_for([v for v in vehicles if v[2] == name], capacities[name])[0]
        zone_raw[name] = zs
        zone_ema[name] = zs if zone_ema[name] is None else (1 - alpha) * zone_ema[name] + alpha * zs
        zone_levels[name] = level_with_hysteresis(zone_ema[name], zone_levels[name])

    # Tiempo que lleva cada zona (y la escena completa) en nivel ALTA
    if level == "ALTA":
        high_since = timestamp if high_since is None else high_since
    else:
        high_since = None
    for name in zones_px:
        if zone_levels[name] == "ALTA":
            zone_high_since[name] = zone_high_since[name] if zone_high_since[name] is not None else timestamp
        else:
            zone_high_since[name] = None
    high_for = 0.0 if high_since is None else timestamp - high_since
    alert_zones = {k: timestamp - v for k, v in zone_high_since.items()
                   if v is not None and timestamp - v >= ALERT_MIN_SECONDS}
    pulse = 0.5 + 0.5 * math.sin(2 * math.pi * 1.5 * timestamp)

    spark.append((timestamp, score_ema))
    while spark and timestamp - spark[0][0] > SPARKLINE_SECONDS:
        spark.popleft()

    # ---- Mapa de calor: acumula donde hay vehículos lentos o detenidos ----
    # (cv2.circle sobrescribe, no suma: se dibuja en una capa y luego se acumula)
    heat *= HEAT_DECAY
    stamp = np.zeros_like(heat)
    for tid, (x1, y1, x2, y2), zone, state, speed in vehicles:
        w = {"DETENIDO": 1.0, "LENTO": 0.35}.get(state, 0.0)
        if w:
            cv2.circle(stamp, ((x1 + x2) // 2 // HEAT_DS, (y1 + 3 * y2) // 4 // HEAT_DS),
                       max(2, (y2 - y1) // (2 * HEAT_DS)), w, -1)
    heat += stamp

    # ---- Dibujo ----
    annotated = frame.copy()
    if SHOW_HEATMAP:
        render_heatmap(annotated, heat, S)
    zone_disp = {k: v for k, v in zone_ema.items()}
    draw_zones(annotated, zones_px, zone_levels, zone_disp, pulse, S)
    for tid, box, zone, state, speed in vehicles:
        if SHOW_TRAILS and len(trails[tid]) > 2:
            draw_trail(annotated, list(trails[tid]), STATE_COLORS[state], S)
        draw_vehicle(annotated, box, state, S)

    n_free = sum(v[3] == "FLUIDO" for v in vehicles)
    n_slow = sum(v[3] == "LENTO" for v in vehicles)
    n_stop = sum(v[3] == "DETENIDO" for v in vehicles)
    draw_alert(annotated, PANEL_X, level, high_for, alert_zones, pulse, S)
    draw_panel(annotated, PANEL_X, S, {
        "t": timestamp, "score": score_ema, "level": level, "n": len(vehicles),
        "n_free": n_free, "n_slow": n_slow, "n_stop": n_stop,
        "zone_scores": zone_disp, "zone_levels": zone_levels, "history": list(spark),
    })

    writer.write(annotated)
    if score_ema > peak[0] and frame_idx > fps:       # ignora el arranque del suavizado
        peak = (score_ema, timestamp, annotated.copy())

    speeds_now = [v[4] for v in vehicles if v[4] is not None]
    row = {
        "frame": frame_idx, "time_s": timestamp, "vehicles": len(vehicles),
        "free_vehicles": n_free, "slow_vehicles": n_slow, "stopped_vehicles": n_stop,
        "stopped_ratio": stopped_ratio, "occupancy_ratio": occupancy, "slowness": slowness,
        "median_speed_bh_s": float(np.median(speeds_now)) if speeds_now else np.nan,
        "congestion_raw": raw_score, "congestion_score": score_ema, "congestion_level": level,
    }
    for name in zones_px:
        row[f"score_{name}"] = zone_ema[name]
        row[f"level_{name}"] = zone_levels[name]
    rows.append(row)

    frame_idx += 1
    pbar.update(1)

pbar.close()
cap.release()
writer.release()

if not rows:
    raise RuntimeError("No se procesaron fotogramas. Verifica el archivo de video.")

metrics = pd.DataFrame(rows)
metrics.to_csv(OUTPUT_CSV, index=False)
if peak[2] is not None:
    cv2.imwrite(OUTPUT_PEAK, peak[2])
print(f"Fotogramas procesados: {len(metrics)}")
print(f"Video: {OUTPUT_VIDEO}")
print(f"CSV: {OUTPUT_CSV}")
print(f"Pico de congestión: {peak[0]:.1f}/100 en t = {peak[1]:.1f} s -> {OUTPUT_PEAK}")


## 7. Resumen y gráficas de congestión

In [ ]:
display(metrics[["vehicles", "slow_vehicles", "stopped_vehicles", "median_speed_bh_s", "congestion_score"]].describe().round(2))

LEVEL_HEX = {"BAJA": "#46c85a", "MODERADA": "#ffb900", "ALTA": "#eb2832"}
zone_cols = [c for c in metrics.columns if c.startswith("score_")]
t = metrics["time_s"]

def shade_levels(ax, levels):
    """Colorea el fondo de la gráfica según el nivel de congestión en cada instante."""
    start = 0
    for i in range(1, len(levels) + 1):
        if i == len(levels) or levels.iloc[i] != levels.iloc[start]:
            ax.axvspan(t.iloc[start], t.iloc[min(i, len(t) - 1)], color=LEVEL_HEX[levels.iloc[start]], alpha=0.12, lw=0)
            start = i

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

# 1) Composición del tráfico por estado
axes[0].stackplot(t, metrics["free_vehicles"], metrics["slow_vehicles"], metrics["stopped_vehicles"],
                  labels=["Fluidos", "Lentos", "Detenidos"],
                  colors=[LEVEL_HEX["BAJA"], LEVEL_HEX["MODERADA"], LEVEL_HEX["ALTA"]], alpha=0.85)
axes[0].plot(t, metrics["vehicles"], color="black", lw=1, label="Total (incluye recién detectados)")
axes[0].set_ylabel("Vehículos")
axes[0].set_title("Composición del tráfico por estado de movimiento")
axes[0].legend(loc="upper left", ncol=4)
axes[0].grid(alpha=0.3)

# 2) Índice global con bandas de nivel
shade_levels(axes[1], metrics["congestion_level"])
axes[1].plot(t, metrics["congestion_raw"], color="gray", lw=0.8, alpha=0.6, label="Índice instantáneo")
axes[1].plot(t, metrics["congestion_score"], color="black", lw=2, label="Índice suavizado")
axes[1].axhline(LIMIT_MODERATE, ls="--", color=LEVEL_HEX["MODERADA"], label="Límite baja/moderada")
axes[1].axhline(LIMIT_HIGH, ls="--", color=LEVEL_HEX["ALTA"], label="Límite moderada/alta")
axes[1].set_ylabel("Índice (0-100)")
axes[1].set_ylim(0, 100)
axes[1].set_title("Índice de congestión global (fondo = nivel detectado)")
axes[1].legend(loc="lower right", ncol=2)
axes[1].grid(alpha=0.3)

# 3) Índice por zona
for c in zone_cols:
    axes[2].plot(t, metrics[c], lw=2, label=c.replace("score_", ""))
axes[2].axhspan(LIMIT_HIGH, 100, color=LEVEL_HEX["ALTA"], alpha=0.10)
axes[2].axhspan(LIMIT_MODERATE, LIMIT_HIGH, color=LEVEL_HEX["MODERADA"], alpha=0.10)
axes[2].axhspan(0, LIMIT_MODERATE, color=LEVEL_HEX["BAJA"], alpha=0.10)
axes[2].set_ylim(0, 100)
axes[2].set_xlabel("Tiempo (s)")
axes[2].set_ylabel("Índice (0-100)")
axes[2].set_title("Índice de congestión por zona")
axes[2].legend(loc="lower right")
axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Nivel promedio:", congestion_label(metrics["congestion_score"].mean()))
print("Congestión promedio:", round(metrics["congestion_score"].mean(), 1), "/ 100")
print("Máximo de vehículos simultáneos:", int(metrics["vehicles"].max()))
pct = metrics["congestion_level"].value_counts(normalize=True).mul(100).round(1)
print("Tiempo en cada nivel (%):", pct.to_dict())
for c in zone_cols:
    z = c.replace("score_", "")
    alta = (metrics[f"level_{z}"] == "ALTA").mean() * 100
    print(f"  {z}: índice medio {metrics[c].mean():.1f}, {alta:.0f}% del tiempo en ALTA")

print("\nFotograma de mayor congestión:")
display(Image(OUTPUT_PEAK, width=900))


## 8. Visualizar el video resultante

In [ ]:
import os

# Convertir el video a formato compatible con HTML5 (H.264/AAC) usando ffmpeg
COMPATIBLE_VIDEO = "/content/congestion_yolo26n_mot_h264.mp4"

print("Convirtiendo el video a formato compatible con navegadores...")
os.system(f"ffmpeg -y -i {OUTPUT_VIDEO} -vcodec libx264 -preset veryfast -pix_fmt yuv420p -f mp4 {COMPATIBLE_VIDEO} -loglevel quiet")

if os.path.exists(COMPATIBLE_VIDEO):
    display(Video(COMPATIBLE_VIDEO, embed=True, width=900))
else:
    print("Error al convertir el video con FFmpeg. Mostrando video original por si acaso:")
    display(Video(OUTPUT_VIDEO, embed=True, width=900))

## 9. Descargar video y métricas

In [ ]:
from google.colab import files
files.download(COMPATIBLE_VIDEO if os.path.exists(COMPATIBLE_VIDEO) else OUTPUT_VIDEO)
files.download(OUTPUT_CSV)
files.download(OUTPUT_PEAK)


## Interpretación y límites

- **MOT / ByteTrack** asigna IDs para no contar cada detección como un vehículo nuevo en cada fotograma. El conteo mostrado es de vehículos visibles en ese instante dentro de las zonas.
- **Velocidad normalizada (bh/s):** el desplazamiento del centroide se divide entre la altura de la caja del vehículo. Así un auto lejano (pequeño) y uno cercano (grande) que avanzan igual obtienen valores comparables. `STOPPED_BH_S` y `SLOW_BH_S` definen DETENIDO y LENTO.
- **Índice de congestión** = `W_OCCUPANCY`·ocupación + `W_SLOWNESS`·lentitud + `W_STOPPED`·proporción detenida. La lentitud compara la mediana de velocidad con `FREE_FLOW_BH_S`. Esto permite detectar congestión de **flujo lento y denso** (tráfico que avanza a vuelta de rueda), no sólo de vehículos totalmente detenidos.
- **Suavizado e histéresis:** `EMA_ALPHA` evita saltos bruscos y `HYSTERESIS` impide que el color parpadee cuando el índice ronda un límite.
- **Zonas:** las coordenadas de `ZONES` están ajustadas al video de la autopista (1280x720). Para otra cámara, redibuja los polígonos (o usa `ZONES = None`) y ajusta `ZONE_CAPACITY` al número de vehículos que caben en cada zona.
- En el video de ejemplo el tráfico es denso y lento durante casi todo el clip (≈25 vehículos, mediana ≈0.7 bh/s, casi ninguno totalmente detenido), por eso domina el nivel ALTA con tramos MODERADA, sobre todo en la calzada derecha.
- Para estimar km/h se necesita calibrar la escena con distancias reales y compensar la perspectiva.
